## Indexing collection

Query performance can be improved by creating indexes over tables of the collection. 
By default, the index creation is minimal to enable faster insertion of data into the collection. 
However, after you've inserted some data, e.g. completed a collection table or created a new detached layer table, you should also create corresponding indexes to enable efficient querying over the collection.
This tutorial shows different kinds of indexes `PostgresStorage` supports and how to create these.

In [1]:
from estnltk import Text
from estnltk.storage.postgres import PostgresStorage, delete_schema
from estnltk.taggers import VabamorfTagger

In [2]:
# Open the storage, and create a new schema
storage = PostgresStorage(dbname='test_db', pgpass_file='~/.pgpass',
                          schema='my_schema', create_schema_if_missing=True)

INFO:storage.py:73: connecting to host: 'localhost', port: '5432', dbname: 'test_db', user: 'postgres'
INFO:storage.py:94: new schema 'my_schema' created
INFO:storage.py:124: schema: 'my_schema', temporary: False, role: 'postgres'


### Creating collection index (for attached layers)

Collection index enables efficient querying over the _attached layers_ inside the collection. 
Recall, attached layers are layers that Text objects possess at the time of the insertion. 

Let's create an example collection with attached layers:

In [3]:
collection = storage.add_collection('collection_with_attached_layers')

with collection.insert() as collection_insert:
    collection_insert(Text('See on esimene lause.').tag_layer("sentences"))
    collection_insert(Text('See on teine lause.').tag_layer("sentences"))
    collection_insert(Text('Sellest lausest ei ole kuulnudki.').tag_layer("sentences"))

INFO:storage.py:260: new empty collection 'collection_with_attached_layers' created
INFO:collection_text_object_inserter.py:104: inserted 3 texts into the collection 'collection_with_attached_layers'


In [4]:
# Browse attached layers
collection

,layer_type,relation_layer,attributes,span_names,ambiguous,sparse,parent,enveloping,meta
words,attached,False,"(normalized_form,)",None,True,False,None,None,[]
sentences,attached,False,(),None,False,False,None,words,[]
compound_tokens,attached,False,"(type, normalized)",None,False,False,None,tokens,[]
tokens,attached,False,(),None,False,False,None,None,[]


Once you've completed the collection, you can create a _[GIN](https://www.postgresql.org/docs/current/gin.html) index_ over the attached layers to enable faster queries over those layers:

In [5]:
collection.create_index()

---
_Creating index before data insertion_. You can also create [GIN](https://www.postgresql.org/docs/current/gin.html) collection index before filling the collection table with data. An example:

```python
# Add new collection table and also create GIN index ( create_index=True )
collection = storage.add_collection('collection_with_attached_layers', create_index=True)

# Fill in the collection with data
with collection.insert() as collection_insert:
    collection_insert(Text('See on esimene lause.').tag_layer("sentences"))
    collection_insert(Text('See on teine lause.').tag_layer("sentences"))
    collection_insert(Text('Sellest lausest ei ole kuulnudki.').tag_layer("sentences"))
```
Be aware that once you create a collection GIN index, inserting data into the collection becomes slower.

---

---

_Removing collection index._ If you need to remove the collection index for some reason, use the method drop_index:

```python
# Add remove attached layer's GIN index from the collection
collection.drop_index()
```

---

In [6]:
# Clean up
storage.delete_collection('collection_with_attached_layers')

### Creating layer index (for detached layer)

Layer index enables efficient querying over a _detached layer_ of the collection. 
Recall, each detached layer is situated in a separate table in the database. 

Let's create an example collection with a detached layer (and its dependency layers):

In [7]:
collection = storage.add_collection('collection_with_detached_layer')

# First, insert texts with dependency layers (attached layers)
with collection.insert() as collection_insert:
    collection_insert(Text('See on esimene lause.').tag_layer("sentences"))
    collection_insert(Text('See on teine lause.').tag_layer("sentences"))
    collection_insert(Text('Sellest lausest ei ole kuulnudki.').tag_layer("sentences"))

INFO:storage.py:260: new empty collection 'collection_with_detached_layer' created
INFO:collection_text_object_inserter.py:104: inserted 3 texts into the collection 'collection_with_detached_layer'


In [8]:
# Add a detached layer to the collection
detached_layer = 'morph_layer'
tagger = VabamorfTagger(disambiguate=False, output_layer=detached_layer)
collection.create_layer(tagger=tagger)

INFO:collection.py:1478: collection: 'collection_with_detached_layer'
INFO:collection.py:1497: preparing to create a new layer: 'morph_layer'
INFO:collection.py:1031: detached layer 'morph_layer' created from template
INFO:collection.py:1534: inserting data into the 'morph_layer' layer table
INFO:collection_detached_layer_inserter.py:87: inserted 3 detached 'morph_layer' layers into the collection 'collection_with_detached_layer'
INFO:collection.py:1584: layer created: 'morph_layer'


In [9]:
# Browse attached layers and a detached layer
collection

,layer_type,relation_layer,attributes,span_names,ambiguous,sparse,parent,enveloping,meta
words,attached,False,"(normalized_form,)",None,True,False,None,None,[]
sentences,attached,False,(),None,False,False,None,words,[]
compound_tokens,attached,False,"(type, normalized)",None,False,False,None,tokens,[]
tokens,attached,False,(),None,False,False,None,None,[]
morph_layer,detached,False,"(normalized_text, lemma, root, root_tokens, ending, clitic, form, partofspeech)",None,True,False,words,None,[]


Once you've completed the detached layer, create a _[GIN](https://www.postgresql.org/docs/current/gin.html) index_ over it to enable faster queries:

In [10]:
# Create GIN index for detached_layer
collection.create_layer_index(detached_layer)

If you have multiple detached layers, a separate index should be created for each detached layer to enable efficient queries.

---
_Creating index before data insertion_. You can also create [GIN](https://www.postgresql.org/docs/current/gin.html) layer index before filling the layer table with data. An example:

```python
# Create a new collection
collection = storage.add_collection('collection_with_detached_layer')

# First, insert texts with dependency layers (attached layers)
with collection.insert() as collection_insert:
    collection_insert(Text('See on esimene lause.').tag_layer("sentences"))
    collection_insert(Text('See on teine lause.').tag_layer("sentences"))
    collection_insert(Text('Sellest lausest ei ole kuulnudki.').tag_layer("sentences"))

# Create tagger
tagger = VabamorfTagger(disambiguate=False)

# Add new layer table, and create a GIN index (create_index=True)
collection.add_layer(tagger.get_layer_template(), create_index=True)

# Fill in the layer with data
collection.create_layer(tagger=tagger, mode='append')
    
```
Be aware that once you create a GIN index for the table, inserting data into the table becomes slower.

---

In [11]:
# Clean up
storage.delete_collection('collection_with_detached_layer')

### Creating layer n-gram index (for detached layer)

#### Creating index 

N-gram index enables to search for n-grams of layer attributes. 
For example, if we create a bigram index on an attribute with values `['see', 'on', 'esimene', 'lause']`, then we can search for two-word phrases, such as *'see-on'*, *'on-esimene'*, *'esimene-lause'*.

Example. First, create a new collection:

In [12]:
collection = storage.add_collection('collection_with_layers')

with collection.insert() as collection_insert:
    collection_insert(Text('See on esimene lause.').tag_layer("sentences"))
    collection_insert(Text('See on teine lause.').tag_layer("sentences"))
    collection_insert(Text('See ei ole midagi.').tag_layer("sentences"))

INFO:storage.py:260: new empty collection 'collection_with_layers' created
INFO:collection_text_object_inserter.py:104: inserted 3 texts into the collection 'collection_with_layers'


To build an ngram index, provide an argument `ngram_index` when creating a new layer. 
`ngram_index` must be a dictionary mapping from attribute names to maximum N value of created N-grams. 
For instance, specifying `ngram_index={"lemma": 2}` creates unigrams and bigrams index on the attribute *lemma* of the *indexed_layer*:

In [13]:
indexed_layer = 'indexed_layer'
tagger = VabamorfTagger(disambiguate=False, output_layer=indexed_layer)

collection.create_layer(tagger=tagger, ngram_index={"lemma": 2})

INFO:collection.py:1478: collection: 'collection_with_layers'
INFO:collection.py:1497: preparing to create a new layer: 'indexed_layer'
INFO:collection.py:1031: detached layer 'indexed_layer' created from template
INFO:collection.py:1173: 'indexed_layer' ngram index table created.
INFO:collection.py:1534: inserting data into the 'indexed_layer' layer table
INFO:collection_detached_layer_inserter.py:87: inserted 3 detached 'indexed_layer' layers into the collection 'collection_with_layers'
INFO:collection.py:1584: layer created: 'indexed_layer'


Note that for storing the _ngrams index_, a separate table is created. 

Created tables can be seen at the `storage`:

In [14]:
storage

After you've completed the layer, you should also create a _[GIN](https://www.postgresql.org/docs/current/gin.html) array index_ over the same ngram columns to enable faster queries:

In [15]:
collection.create_layer_index( indexed_layer, index_type='ngram_index', ngram_index={"lemma": 2} )

---

_Creating index before data insertion_. You can also create [GIN](https://www.postgresql.org/docs/current/gin.html) array index over the ngram columns before filling the table with data. An example:

```python
# Add new layer table, ngram index and a GIN array index over the ngram index (create_ngram_index=True)
collection.add_layer(tagger.get_layer_template(), ngram_index={"lemma": 2}, create_ngram_index=True)

# Fill in the layer with data
collection.create_layer(tagger=tagger, ngram_index={"lemma": 2}, mode='append')
```

Be aware that once you create a GIN index for the table, inserting data into the table becomes slower.

---

#### Querying index 

To search an ngram index, use `LayerNgramQuery` query:

Search entries containing lemma bigram 'see-olema':

In [16]:
from estnltk.storage.postgres import LayerNgramQuery

q = LayerNgramQuery( { indexed_layer: {
        "lemma": [("see", "olema")]
    }})
for key, text in collection.select(query=q):
    print(key, text)

0 Text(text='See on esimene lause.')
1 Text(text='See on teine lause.')


Search 'teine-lause' OR 'olema-esimene':

In [17]:
q = LayerNgramQuery( { indexed_layer: {
        "lemma":  [("teine", "lause"), ("olema", "esimene")]
    }})
for key, text in collection.select(query=q):
    print(key, text)

0 Text(text='See on esimene lause.')
1 Text(text='See on teine lause.')


Search 'see-olema' AND 'olema-esimene':

In [18]:
q = LayerNgramQuery( { indexed_layer: {
        "lemma":  [[("see", "olema"), ("olema", "esimene")]]
    }})
for key, text in collection.select(query=q):
    print(key, text)

0 Text(text='See on esimene lause.')


<p>
<div class="alert alert-block alert-warning">
<h4><i>Limitations of ngram indexing</i></h4> 
<p>Currently, ngram indexing cannot be applied on creating relation layers.</p>
</div>
</p>

<p>
<div class="alert alert-block alert-warning">
<h4><i>Version differences: EstNLTK v1.7.4 and earlier versions versus EstNLTK v1.7.5</i></h4> 
<p>In EstNLTK v1.7.4 and earlier versions, layer ngram index columns were stored as extra columns of the detached layer table. This has been changed in version 1.7.5, in which a separate table (with a table name suffix <code>__ngrams</code>) is created for storing layer ngram index columns. Note that there is no backwards compatibility between the versions: <code>LayerNgramQuery</code> of EstNLTK v1.7.4 and earlier versions cannot be used on ngram indexes created with EstNLTK v1.7.5 (and vice versa).</p>
</div>
</p>

In [19]:
storage.delete_collection( collection.name )

In [20]:
# Clean up
delete_schema(storage)
storage.close()

---

## Technical notes

### Changes in index naming ( v1.7.5 ) 

The main difference in index naming between v1.7.5 and earlier versions is that the index name is constructed via concatenating prefix `'idx_sha1_'` and a SHA-1 hexdigest of the index pattern name. 
In the new version, index names have a fixed-length (49 chars), assuring that they meet [Postgres's identifier name length limit](https://www.postgresql.org/docs/current/datatype-character.html#DATATYPE-CHARACTER-SPECIAL-TABLE).

There are also some more subtle naming differences, see the table below for a comparison of the naming patterns.

| Indexed table | Index naming in old versions (v1.7.4 and earlier)  |  Index naming in v1.7.5 |
|-----------| ----------- | ----------- |
| Collection table | `'idx_{collection_name}__layer_data'`     | `'idx_sha1_'+SHA1('{collection_name}__layer_data')` |
| Collection table | `'idx_{collection_name}__relation_layer_data'`  |  `'idx_sha1_'+SHA1('{collection_name}__relation_layer_data')` |
| Detached layer table | `'idx_{collection_name}__{layer_name}__layer__text_id'`     | `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__layer__text_id')` |
| Detached layer table | `'idx_{collection_name}__{layer_name}__layer_spans'`     | `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__layer_spans')` |
| Detached layer table | `'idx_{collection_name}__{layer_name}__layer_relations'`     | `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__layer_relations')` |
| Fragmented layer table | `'idx_{collection_name}__{layer_name}__fragment__text_id'`     | `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__fragment__text_id')` |
| Fragmented layer table | `'idx_{collection_name}__{layer_name}__fragment_spans'`     | `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__fragment_spans')` |
| Fragmented layer table | `'idx_{collection_name}__{layer_name}__fragment_relations'`     | `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__fragment_relations')` |
| Layer N-grams table |      | `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__ngrams__text_id')` |
| Layer N-grams table | `'idx_{collection_name}__{layer_name}__layer_{ngram_column_name}'` | `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__ngrams_{ngram_column_name}')` |
| Layer N-grams table | `'idx_{collection_name}__{layer_name}__fragment_{ngram_column_name}'`     | `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__ngrams_{ngram_column_name}')`  |


### PostgresStorage's new indexing behaviour ( v1.7.5 ) 

* Collection tables are indexed in the following way:
   * Once a new collection table is created via `storage.add_collection(...)`:
        * no indexes are created by default;
        * optionally, if `create_index` is set, then 2 indexes are created over its attached layers:
           * `'idx_sha1_'+SHA1('{collection_name}__layer_data')` -- [GIN index](https://www.postgresql.org/docs/current/gin.html) over attached span layers (`(data->'layers') jsonb_path_ops`);
           * `'idx_sha1_'+SHA1('{collection_name}__relation_layer_data')` -- GIN index over attached relation layers (`(data->'relation_layers') jsonb_path_ops`);
    * Alternatively, the attached layer indexes can be created later via `collection.create_index()` method and, `collection.drop_index()` can be used to remove the indexes;
* Detached layer tables are indexed in the following way:
   * Once a new detached layer is created via `collection.add_layer(...)`:
        * an index over `text_id` is always created:
            * `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__layer__text_id')` -- index over `text_id` column;
        * optionally, if `create_index` is set, then an index over spans or relations is created:
            * `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__layer_spans')` -- GIN index over layer's spans (`(data->'spans') jsonb_path_ops`) (if the layer is span layer);
            * `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__layer_relations')` -- GIN index over layer's relations (`(data->'relations') jsonb_path_ops`) (if the layer is relation layer);
        * optionally, if `ngram_index` is set:
            * an index over `text_id` is always created:
                * `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__ngrams__text_id')` -- index over `text_id` column in the layer ngrams table;
            * if `create_ngram_index` is also set, then an array index is created:
                * `'idx_sha1_'+SHA1('{collection_name}__{layer_name}__ngrams_{ngram_column_name}')` -- GIN index over n-gram array columns in the layer ngrams table;
    * Alternatively, you can use `collection.create_layer_index(...)` method to add detached layer indexes later:
        * `collection.create_layer_index(layer_name, index_type='data')` -- creates an index over layer's spans or relations;
        * `collection.create_layer_index(layer_name, index_type='ngram_index', ngram_index={'ngram_column_name': n})` -- creates an index over n-gram data in the layer ngrams table;

### PostgresStorage's old indexing behaviour (v1.7.4) 

* Collection tables are indexed in the following way:
   * Once a new collection table is created via `storage.add_collection`, 2 indexes are created over its attached layers:
        * `'idx_{collection_name}_layer_data'` -- [GIN index](https://www.postgresql.org/docs/current/gin.html) over attached span layers (`(data->'layers') jsonb_path_ops`);
        * `'idx_{collection_name}_relation_layer_data'` -- GIN index over attached relation layers (`(data->'relation_layers') jsonb_path_ops`);
   * Alternatively, the same attached layer indexes can be created via `collection.create_index()` method and, `collection.drop_index()` can be used to remove the indexes;
* Detached layer tables are indexed in the following way:
   * Once a new detached layer is created via `collection.add_layer()`:
        * an index over `text_id` is always created:
            * `'idx_{detached_layer_table_name}__text_id'` -- index over `text_id` column;
        * optionally, if `create_index` is set, then an index over spans or relations is created:
            * `'idx_{detached_layer_table_name}_spans'` -- GIN index over layer's spans (`(data->'spans') jsonb_path_ops`) (if the layer is span layer);
            * `'idx_{detached_layer_table_name}_relations'` -- GIN index over layer's relations (`(data->'relations') jsonb_path_ops`) (if the layer is relation layer);
        * optionally, if `ngram_index` is set, then an array index is created:
            * `'idx_{detached_layer_table_name}_{ngram_column_name}'` -- GIN index over n-gram array column;